---
title: "Module 2 - Tutorial 1"
subtitle: "From SEC HTML to a Reusable Section Corpus"
number-sections: true
date: "2026-02-01"
date-modified: today
date-format: long
format:
    html:
        theme: [cosmo, ../theme/metanalytics.scss]
        code-overflow: wrap
        toc: true
        toc-depth: 3
        number-sections: true
        df-print: paged
    docx:
        reference-doc: ../theme/APA7.docx
bibliography: ../references.bib
csl: ../mis-quarterly.csl
engine: jupyter
execute:
    echo: true
    eval: false
    warning: false
    message: false
categories: ['M02:', 'Tutorial', 'SEC HTML', 'Corpus Construction']
description: "Tutorial for Module 2 focused on turning messy SEC HTML into a clean reusable corpus for later embedding and retrieval work."
---

# Module Learning Objectives

By the end of Module 2, students should be able to:

1. transform messy SEC source documents into structured corpora
2. save reusable section-level artifacts for later modules
3. validate extraction quality rather than only generating output files
4. prepare text data for representation learning and retrieval

## Tutorial Learning Objectives

By the end of this tutorial, students should be able to:

- discover and load SEC HTML filings
- normalize and clean noisy filing text
- extract target sections into a structured dataframe
- save a reusable, auditable corpus for later module work
- estimate simple n-gram probabilities from extracted SEC text

## How This Tutorial Connects

- `M02_Lab1` validates extraction and cleaning decisions on a smaller practice sample and saves reusable artifacts.
- `M02_Lab2` works from chunked SEC text and compares sparse vs dense semantic representations.
- `M02_A` does not reuse this tutorial output directly, but it does reuse the same discipline around auditable artifacts, chunking, and reproducible notebook workflow.
- `M02_participation1` and `M02_participation2` can reuse the same corpus-building logic.

::: {.callout-tip}
This tutorial teaches the mechanics. The lab asks you to verify that the pipeline is trustworthy, and the assignment asks you to apply related ideas in your own executed business analysis.
:::

# Recommended assignment build sequence

Treat this as a data engineering workflow:

1. Discover and load HTML filings
2. Clean HTML into normalized text
3. Detect and extract Item 1, Item 1A, and Item 7
4. Build a structured corpus dataframe
5. Save outputs for later modules

![Bayesian Modeling](./M02_lecture01_figures/baysian-modeling.png){width=80% fig-align="center" #fig-baysian-modeling fig-alt="Recommended build sequence for the assignment using Bayesian modeling steps"}

# Discover and load HTML filings

## Importing necessary libraries

In [ ]:
import os
import re
import random
import json
import unicodedata
from pathlib import Path
from typing import Optional, Dict, List

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

## Define the data path and discover HTML files

Use either:

- `../data/SEC-10K-2024-HTML`
- a mounted Google Drive folder containing the SEC HTML corpus

In [ ]:
DATA_DIR = Path("../data/SEC-10K-2024-HTML")
OUTPUT_DIR = Path("../data/outputs/sec_10k_sections")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_HTML_FILES = sorted(DATA_DIR.rglob("*.html")) + sorted(DATA_DIR.rglob("*.htm"))
print(f"Total HTML files found: {len(ALL_HTML_FILES)}")

For Google Colab:

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/data/SEC-10K-2024-HTML")
OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/data/outputs/sec_10k_sections")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Select a small sample first

Start small for debugging, then scale up for the assignment.

In [ ]:
def select_html_files(
    files: List[Path],
    sample_n: Optional[int] = None,
    random_sample: bool = False,
    random_state: int = 42
) -> List[Path]:
    if sample_n is None:
        return files

    sample_n = min(sample_n, len(files))
    if random_sample:
        rng = random.Random(random_state)
        return sorted(rng.sample(files, sample_n))
    return files[:sample_n]

In [ ]:
html_files = select_html_files(ALL_HTML_FILES, sample_n=25, random_sample=False)
print(f"Files selected: {len(html_files)}")
html_files[:5]

# HTML cleaning helpers

In [ ]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\xa0", " ")
    text = text.replace("&nbsp;", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "ix:header", "header", "footer"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    return normalize_text(text)

# Extract target SEC sections

In Module 2, the key idea is that extraction must be **deterministic enough to audit**.

In [ ]:
ITEM_PATTERNS = {
    "item_1": re.compile(r"item\s+1[^a-z0-9].*?(?=item\s+1a[^a-z0-9]|item\s+2[^a-z0-9])", re.I | re.S),
    "item_1a": re.compile(r"item\s+1a[^a-z0-9].*?(?=item\s+1b[^a-z0-9]|item\s+2[^a-z0-9])", re.I | re.S),
    "item_7": re.compile(r"item\s+7[^a-z0-9].*?(?=item\s+7a[^a-z0-9]|item\s+8[^a-z0-9])", re.I | re.S),
}


def extract_sections(clean_text: str) -> Dict[str, Optional[str]]:
    out = {}
    for name, pattern in ITEM_PATTERNS.items():
        match = pattern.search(clean_text)
        out[name] = match.group(0).strip() if match else None
    return out

# Build a structured corpus table

In [ ]:
records = []

for path in html_files:
    raw_html = path.read_text(encoding="utf-8", errors="ignore")
    clean_text = html_to_text(raw_html)
    sections = extract_sections(clean_text)

    records.append({
        "file_name": path.name,
        "clean_char_count": len(clean_text),
        "item_1_len": len(sections["item_1"]) if sections["item_1"] else 0,
        "item_1a_len": len(sections["item_1a"]) if sections["item_1a"] else 0,
        "item_7_len": len(sections["item_7"]) if sections["item_7"] else 0,
        "item_1": sections["item_1"],
        "item_1a": sections["item_1a"],
        "item_7": sections["item_7"],
    })

corpus_df = pd.DataFrame(records)
corpus_df.head()

Run the usual checks:

In [ ]:
corpus_df.info()
corpus_df.describe(include="all")

# From extracted SEC text to n-gram language modeling

Once the SEC filing text has been cleaned and structured, it can serve as a small language-modeling corpus. The Apple example in [help-code/colab_tokenization_demo.ipynb](/D:/Repositories/AD698-generative-ai-for-BA/help-code/colab_tokenization_demo.ipynb) is a good bridge from corpus engineering to probabilistic language modeling because it shows how token counts become unigram, bigram, and trigram probabilities.

## A short Apple 10-K example

Use a short sentence from the Apple 10-K and compare it with Spanish and German translations:

In [ ]:
apple_sentence_en = "The Company designs, manufactures and markets mobile communication and media devices and personal computers."
apple_sentence_es = "La empresa diseña, fabrica y comercializa dispositivos móviles de comunicación y medios y computadoras personales."
apple_sentence_de = "Das Unternehmen entwickelt, produziert und vermarktet mobile Kommunikations- und Mediengeräte sowie Personal Computer."

print(apple_sentence_en)
print(apple_sentence_es)
print(apple_sentence_de)

The teaching point is that the **meaning** is similar across languages, but the **token order** and local conditional probabilities are language-specific.

## Tokenize and build n-grams

In [ ]:
def tokenize_words(text: str) -> list[str]:
    return re.findall(r"[a-z]+", text.lower())


def build_ngram_tables(tokens: list[str]):
    unigrams = Counter(tokens)
    bigrams = Counter(zip(tokens[:-1], tokens[1:]))
    trigrams = Counter(zip(tokens[:-2], tokens[1:-1], tokens[2:]))
    trigram_prefixes = Counter((w1, w2) for w1, w2, _ in trigrams.elements())
    return unigrams, bigrams, trigrams, trigram_prefixes

In [ ]:
apple_demo_text = " ".join([
    "The Company designs, manufactures and markets mobile communication and media devices and personal computers.",
    "The Company sells its products worldwide through its retail stores, online stores and direct sales force.",
    "The business, financial condition and operating results of the Company can be affected by a number of factors."
])

tokens_en = tokenize_words(apple_demo_text)
unigrams_en, bigrams_en, trigrams_en, trigram_prefixes_en = build_ngram_tables(tokens_en)

print(f"Total English tokens: {len(tokens_en)}")
print("Top 10 bigrams:")
print(bigrams_en.most_common(10))

## Calculate conditional probabilities

In [ ]:
def bigram_prob(bigrams: Counter, unigrams: Counter, w1: str, w2: str, alpha: float = 1.0) -> float:
    vocab_size = len(unigrams)
    return (bigrams[(w1, w2)] + alpha) / (unigrams[w1] + alpha * vocab_size)


def trigram_prob(trigrams: Counter, trigram_prefixes: Counter, unigrams: Counter, w1: str, w2: str, w3: str, alpha: float = 1.0) -> float:
    vocab_size = len(unigrams)
    prefix = trigram_prefixes[(w1, w2)]
    return (trigrams[(w1, w2, w3)] + alpha) / (prefix + alpha * vocab_size)

In [ ]:
print("Selected probabilities from the Apple business-text corpus:")
print("P(company | the) =", round(bigram_prob(bigrams_en, unigrams_en, "the", "company"), 4))
print("P(designs | company) =", round(bigram_prob(bigrams_en, unigrams_en, "company", "designs"), 4))
print("P(operating | financial, condition) =", round(trigram_prob(trigrams_en, trigram_prefixes_en, unigrams_en, "financial", "condition", "operating"), 4))

## Compare fluent English with translation-like word order

A good English language model should prefer fluent English token order over awkward literal-order alternatives.

In [ ]:
def sentence_log_prob(sentence: str, bigrams: Counter, unigrams: Counter, alpha: float = 1.0) -> float:
    words = tokenize_words(sentence)
    score = 0.0
    for i in range(1, len(words)):
        score += np.log(bigram_prob(bigrams, unigrams, words[i - 1], words[i], alpha))
    return score

In [ ]:
english_fluent = "the company sells its products worldwide through its retail stores"
english_spanish_literal = "the company sells worldwide its products through its retail stores"
english_german_literal = "the company through its retail stores sells its products worldwide"

for label, sent in [
    ("Fluent English", english_fluent),
    ("Spanish-style literal order", english_spanish_literal),
    ("German-style literal order", english_german_literal),
]:
    print(label, "->", round(sentence_log_prob(sent, bigrams_en, unigrams_en), 3))

## Optional multilingual extension

Students can also build separate n-gram models for English, Spanish, and German translations and then compare generated continuations in each language:

In [ ]:
multilingual_corpora = {
    "english": [
        "the company designs and markets mobile devices",
        "the company sells products through retail stores",
        "the company offers software and services",
    ],
    "spanish": [
        "la empresa diseña y comercializa dispositivos moviles",
        "la empresa vende productos a traves de tiendas minoristas",
        "la empresa ofrece software y servicios",
    ],
    "german": [
        "das unternehmen entwickelt und vermarktet mobile gerate",
        "das unternehmen verkauft produkte uber einzelhandelsgeschafte",
        "das unternehmen bietet software und dienstleistungen an",
    ],
}

In [ ]:
def generate_from_bigrams(sentences: list[str], start_word: str, n_words: int = 8):
    tokens = []
    for s in sentences:
        tokens.extend(tokenize_words(s))
    unigrams, bigrams, _, _ = build_ngram_tables(tokens)
    vocab = sorted(unigrams.keys())
    current = start_word
    output = [current]
    for _ in range(n_words - 1):
        scores = np.array([bigram_prob(bigrams, unigrams, current, w) for w in vocab])
        scores = scores / scores.sum()
        current = np.random.choice(vocab, p=scores)
        output.append(current)
    return " ".join(output)

In [ ]:
for lang, sentences in multilingual_corpora.items():
    start = {"english": "the", "spanish": "la", "german": "das"}[lang]
    print(lang, "->", generate_from_bigrams(sentences, start_word=start))

This extension helps students see that **n-gram generation is local, language-specific, and highly sensitive to corpus style and token order**.

# Save outputs for later modules

In [ ]:
corpus_df.to_csv(OUTPUT_DIR / "sec_section_corpus_sample.csv", index=False)

# What to validate before you move on

Students should inspect:

1. missing-section counts
2. section-length distributions
3. one manually reviewed extraction example per target section
4. whether the saved corpus can be reused directly in `M02_A`

# Module alignment

- `M02_LN1` explains why language modeling depends on structured text representations.
- `M02_LN2` moves from words to document structure and corpus engineering.
- `M02_Lab1` should test your extraction quality and artifact discipline.
- `M02_A` should scale this exact pipeline without changing the logic.

The big idea for Module 2 is:  
`messy SEC HTML -> normalized sections -> reusable dataframe -> saved corpus for downstream modeling`

## AI Use and Share Links {.unnumbered}

If generative AI materially supports your work for this tutorial, include an AI disclosure appendix or separate AI disclosure document if your instructor requests one. Include the complete prompt(s), relevant output excerpt(s), validation steps, and direct shared chat link(s) when available.

![](../M0/M0_lecture01_figures/chatgpt-share.jpg){width="80%" fig-align="center"}

![](../M0/M0_lecture01_figures/gemini-share-conversation.png){width="80%" fig-align="center"}

When possible, use **one continuous chat** for the activity so the full reasoning trail can be reviewed in one place. If you used multiple chats, share **all chats directly related** to the work.

Tools such as **ChatGPT**, **Claude**, and **Gemini** support direct chat sharing. If sharing or export is not available, include copied prompt/output evidence or screenshots instead. Because shared links may be viewable by anyone with the link, do not include confidential, personal, or restricted information in those chats.
